#Morphological Analysis Due To Osmotic Stress

In [1]:
#Importing Required Libraries And Setting Up Visualization Style
import numpy as np,pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu,ttest_ind,levene,shapiro
import warnings

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)
plt.rcParams["font.size"]=10

In [ ]:
#Loading Dataset
def load_data(data_path):
    fpath=f"{data_path}/41592_2025_2685_MOESM10_ESM.xlsx"
    print("Loading  Osmotic Stress Dataset")
    df=pd.read_excel(fpath,sheet_name='Cell statistics')
    print(f"Dataset Loaded: {df.shape[0]} Rows × {df.shape[1]} Columns")
    print(f"\nConditions: {df['Condition'].unique()}")
    print(f"\nSample Sizes:")
    print(df['Condition'].value_counts())
    return df

data_path="/content/drive/MyDrive/Organoid Analysis"
df_fig3=load_data(data_path)

Hypothesis Testing

* H0 (Null): Hypertonic medium has no effect on cell morphology

* H1 (Alternative): Hypertonic medium causes significant changes in cell volume, shape, and chromatin organization

Experimental Design

* 2 conditions: Isotonic (control) vs Hypertonic (stressed)

* 2 wells total (1 per condition)

* 6 imaging fields per condition

* Total cells: ~2,062 (unbalanced between conditions)


In [ ]:
def summarize_exp_dsgn(df):
    print("Experimental Design Summary")
    print("\nSample Structure:")
    print(f"Total Cells: {len(df)}")
    print("\nBreakdown By Condition:")
    for i in df['Condition'].unique():
        condn_df=df[df['Condition']==i]
        n_wells=condn_df['Well'].nunique()
        n_fields=condn_df['Field'].nunique()
        n_cells=len(condn_df)

        print(f"\n{i}:")
        print(f">Wells: {n_wells}")
        print(f">Fields: {n_fields}")
        print(f">Total Cells: {n_cells}")

    #Getting Numeric Feature Columns
    feat_cols=[i for i in df.columns if 'Value of Default' in i]
    print(f"\nMorphological Features For Analysis =>({len(feat_cols)}):")
    for i in feat_cols:
        #Cleaning Up Column Name For Display
        feat=i.replace('Value of Default','').replace('_',' ')
        print(f">{feat}")
    return feat_cols

feat_cols=summarize_exp_dsgn(df_fig3)

In [ ]:
#Comparing Distributions Of Each Morphological Feature Visually
def feat_cmprsn_histgms(df,feat_cols,condition_col='Condition'):
    n_features=len(feat_cols)
    n_cols=2
    n_rows=(n_features+1)//2

    fig,axes=plt.subplots(n_rows,n_cols,figsize=(14,4*n_rows))
    axes=axes.flatten()

    conditions=df[condition_col].unique()
    colors=['#3498db','#e74c3c']  #Blue For Isotonic::Red For Hypertonic
    for idx,col in enumerate(feat_cols):
        ax=axes[idx]
        for i,j in zip(conditions,colors):
            data=df[df[condition_col]==i][col].dropna()
            ax.hist(data,bins=30,alpha=0.6,label=i,color=j,edgecolor='black')

        #Cleaning Up Feature Name For Title
        feat=col.replace('Value of Default','').replace('_',' ')
        ax.set_xlabel(feat,fontsize=10)
        ax.set_ylabel('Frequency',fontsize=10)
        ax.set_title(feat,fontsize=11)
        ax.legend()
        ax.grid(True,alpha=0.3)

    #Hiding Unused Subplots
    for i in range(n_features,len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.suptitle('Feature Distributions: Isotonic Vs Hypertonic',
                 fontsize=16,y=1.001)
    plt.show()

feat_cmprsn_histgms(df_fig3,feat_cols)

In [ ]:
def plt_cmprsn_violins(df,feat_cols,condition_col='Condition'):
    n_features=len(feat_cols)
    n_cols=3
    n_rows=(n_features+2)//3
    fig,axes=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))
    axes=axes.flatten()
    for i,j in enumerate(feat_cols):
        ax=axes[i]
        #Creating Violin Plots
        sns.violinplot(data=df,x=condition_col,y=j,ax=ax,
                      palette=['#3498db','#e74c3c'],inner='box')
        #Cleaning Up Feature Name
        feat=j.replace('Value of Default','').replace('_',' ')
        ax.set_xlabel('')
        ax.set_ylabel(feat,fontsize=10)
        ax.set_title(feat,fontsize=11)
        ax.grid(True,alpha=0.3,axis='y')

    #Hiding Unused Subplots
    for i in range(n_features,len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.suptitle('Violin Plots: Isotonic Vs Hypertonic',
                 fontsize=16,y=1.001)
    plt.show()

plt_cmprsn_violins(df_fig3,feat_cols)

In [ ]:
def plt_cmprsn_bxs(df,feat_cols,condition_col='Condition'):
    n_features=len(feat_cols)
    n_cols=3
    n_rows=(n_features+2)//3
    fig,axes=plt.subplots(n_rows,n_cols,figsize=(15,4*n_rows))
    axes=axes.flatten()

    for i,j in enumerate(feat_cols):
        ax=axes[i]
        #Creating Box Plot
        sns.boxplot(data=df,x=condition_col,y=j,ax=ax,
                   palette=['#3498db','#e74c3c'],width=0.5)
        #Overlaying Strip Plot With Reduced Point Size
        sns.stripplot(data=df,x=condition_col,y=j,ax=ax,
                     color='black',alpha=0.2,size=1,jitter=True)
        #Cleaning Up Feature Name
        feat=j.replace('Value of Default','').replace('_',' ')
        ax.set_xlabel('')
        ax.set_ylabel(feat,fontsize=10)
        ax.set_title(feat,fontsize=11)
        ax.grid(True,alpha=0.3,axis='y')

    #Hiding Unused Subplots
    for i in range(n_features,len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.suptitle('Box Plots With Data Points: Isotonic Vs Hypertonic',
                fontsize=16,y=1.001)
    plt.show()

plt_cmprsn_bxs(df_fig3,feat_cols)

Statistical Testing

In [ ]:
def normality(df,feat_cols,condition_col='Condition',sample_size=5000):
    res=[]
    for i in feat_cols:
        for j in df[condition_col].unique():
            data=df[df[condition_col]==j][i].dropna()
            #Using Subset If Sample Is Large
            if len(data)>sample_size:
                data=data.sample(sample_size,random_state=42)
            #Shapiro-Wilk Testing
            stat,p_val=shapiro(data)
            is_normal=p_val>0.05
            res.append({
                'Feature':i.replace('Value Of Default',''),
                'Condition':j,
                'P_Value':p_val,
                'Is_Normal':'Yes' if is_normal else 'No'
            })
    res_df=pd.DataFrame(res)
    #Summary
    print(f"\nNormality Test Results (α=0.05):")
    print(f"Features Passing Normality: {(res_df['Is_Normal']=='Yes').sum()}/{len(res_df)}")
    return res_df

normality_res=normality(df_fig3,feat_cols)
print("\n",normality_res)

In [ ]:
def check_equal_var(df,feat_cols,condition_col='Condition'):
    res=[]
    condn=df[condition_col].unique()

    for i in feat_cols:
        groups=[df[df[condition_col]==j][i].dropna() for j in condn]

        #Levene's Test
        stat,p_val=levene(*groups)
        equal_var=p_val>0.05
        res.append({
            'Feature':i.replace('Value of Default',''),
            'P_Value':p_val,
            'Equal_Variance':'Yes' if equal_var else 'No'
        })
    res_df=pd.DataFrame(res)
    # Summary
    print(f"\nEqual Variance Test Results (α=0.05):")
    print(f"Features With Equal Variance: {(res_df['Equal_Variance']=='Yes').sum()}/{len(res_df)}")
    return res_df

var_res=check_equal_var(df_fig3,feat_cols)
print("\n",var_res)

In [ ]:
def calc_cohens_d(grp1,grp2):
    n1,n2=len(grp1),len(grp2)
    var1,var2=np.var(grp1,ddof=1),np.var(grp2,ddof=1)
    #Pooled Standard Deviation
    pooled_std=np.sqrt(((n1-1)*var1+(n2-1)*var2)/(n1+n2-2))
    #Cohen's d
    d=(np.mean(grp1)-np.mean(grp2))/pooled_std
    return d


def stats_tests(df,feat_cols,condition_col='Condition'):
    condn=sorted(df[condition_col].unique())
    g1,g2=condn[0],condn[1]
    res=[]
    for i in feat_cols:
        #Getting Data For Both Groups
        grp1=df[df[condition_col]==g1][i].dropna()
        grp2=df[df[condition_col]==g2][i].dropna()
        #Descriptive Statistics
        mean1,mean2=grp1.mean(),grp2.mean()
        std1,std2=grp1.std(),grp2.std()
        n1,n2=len(grp1),len(grp2)
        #Checking Assumptions
        x,p_norm1=shapiro(grp1.sample(min(5000,len(grp1)),random_state=42))
        x,p_norm2=shapiro(grp2.sample(min(5000,len(grp2)),random_state=42))
        x,p_var=levene(grp1,grp2)
        assumption=(p_norm1>0.05) and (p_norm2>0.05) and (p_var>0.05)
        #Choosing Test
        if assumption:
            stat,p_val=ttest_ind(grp1,grp2)
            test_used="t-test"
        else:
            stat,p_val=mannwhitneyu(grp1,grp2,alternative='two-sided')
            test_used="Mann-Whitney U"

        #Calculating Effect Size (Cohen's d)
        cohens_d=calc_cohens_d(grp1,grp2)
        #Effect Size Interpretation
        if abs(cohens_d)<0.2:
            effect_mag="Negligible"
        elif abs(cohens_d)<0.5:
            effect_mag="Small"
        elif abs(cohens_d)<0.8:
            effect_mag="Medium"
        else:
            effect_mag="Large"
        res.append({
            'Feature':i.replace('Value of Default',''),
            f'{g1}_Mean':mean1,
            f'{g1}_SD':std1,
            f'{g1}_N':n1,
            f'{g2}_Mean':mean2,
            f'{g2}_SD':std2,
            f'{g2}_N':n2,
            'Mean_Difference':mean2-mean1,
            'Test_Used':test_used,
            'P_value':p_val,
            'Cohens_d':cohens_d,
            'Effect_Size':effect_mag
        })
    res_df=pd.DataFrame(res)

    #Applying Bonferroni Correction
    alpha=0.05
    bonferroni_threshold =alpha/len(feat_cols)
    res_df['Significant_Bonferroni']=res_df['P_value']<bonferroni_threshold
    print(f"\nStatistical Tests Complete")
    print(f"Bonferroni-Corrected Significance Threshold: {bonferroni_threshold:.4f}")
    print(f"Significant Features: {res_df['Significant_Bonferroni'].sum()}/{len(feat_cols)}")
    return res_df

test_res=stats_tests(df_fig3,feat_cols)

In [ ]:
def display_test_res(res_df):
    #Selecting Key Columns For Display
    display_cols=['Feature','Mean_Difference','P_value','Cohens_d',
                   'Effect_Size','Test_Used','Significant_Bonferroni']
    display_df=res_df[display_cols].copy()
    #Formating Numeric Columns
    display_df['Mean_Difference']=display_df['Mean_Difference'].round(3)
    display_df['P_value']=display_df['P_value'].apply(lambda x: f"{x:.4f}")
    display_df['Cohens_d']=display_df['Cohens_d'].round(3)
    print(display_df.to_string(index=False))
    #Saving To CSV
    res_df.to_csv('osmotic_stress_statistical_results.csv',index=False)
    print("\nFull Results Saved To 'osmotic_stress_statistical_results.csv'")

display_test_res(test_res)

In [ ]:
def plt_effect_sizes(res_df):
    #Sorting By Absolute Effect Size
    plt_df=res_df.sort_values('Cohens_d',key=abs,ascending=True)
    fig,ax=plt.subplots(figsize=(10,8))
    #Coloring By Significance
    colors=['#e74c3c' if i else '#95a5a6'
             for i in plt_df['Significant_Bonferroni']]
    #Creating Horizontal Bar Plot
    y_pos=np.arange(len(plt_df))
    ax.barh(y_pos,plt_df['Cohens_d'],color=colors,edgecolor='black',linewidth=0.5)
    #Adding Reference Lines For Effect Size Thresholds
    ax.axvline(x=0,color='black',linestyle='-',linewidth=1)
    ax.axvline(x=0.2,color='gray',linestyle='--',linewidth=0.8,alpha=0.5,label='Small (0.2)')
    ax.axvline(x=-0.2,color='gray',linestyle='--',linewidth=0.8,alpha=0.5)
    ax.axvline(x=0.5,color='gray',linestyle='--',linewidth=0.8,alpha=0.5,label='Medium (0.5)')
    ax.axvline(x=-0.5,color='gray',linestyle='--',linewidth=0.8,alpha=0.5)
    ax.axvline(x=0.8,color='gray',linestyle='--',linewidth=0.8,alpha=0.5,label='Large (0.8)')
    ax.axvline(x=-0.8,color='gray',linestyle='--',linewidth=0.8,alpha=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plt_df['Feature'])
    ax.set_xlabel("Cohen's d (Effect Size)",fontsize=12)
    ax.set_title("Effect Sizes: Hypertonic Vs Isotonic\n(Red->Significant After Bonferroni Correction)",
                fontsize=14)
    ax.legend(loc='lower right')
    ax.grid(True,alpha=0.3,axis='x')
    plt.tight_layout()
    plt.show()

plt_effect_sizes(test_res)

Examining Field-Level Variation

In [ ]:

def analyze_field_variation(df,feat_cols,condition_col='Condition',field_col='Field'):
    #Calculating Mean Per Field For Each Feature
    field_means=df.groupby([condition_col,field_col])[feat_cols].mean().reset_index()
    #Calculating Coefficient Of Variation Across Fields For Each Condition
    cv_res=[]
    for i in feat_cols:
        for j in df[condition_col].unique():
            field_val=field_means[field_means[condition_col]==j][i]
            mean_val=field_val.mean()
            std_val=field_val.std()
            cv=(std_val/mean_val)*100 if mean_val!=0 else 0
            cv_res.append({
                'Feature':i.replace('Value of Default',''),
                'Condition':j,
                'Field_CV_%':cv,
                'Field_Mean':mean_val,
                'Field_SD':std_val
            })
    cv_df=pd.DataFrame(cv_res)
    print("\nField-To-Field Variability (Coefficient Of Variation %):")
    print(cv_df.pivot(index='Feature',columns='Condition',values='Field_CV_%').round(2))
    return field_means,cv_df

field_means,field_cv=analyze_field_variation(df_fig3,feat_cols)

In [ ]:
def plt_field_consistency(df,feat_col,condition_col='Condition',field_col='Field'):
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    conditions=sorted(df[condition_col].unique())
    for i,j in enumerate(conditions):
        ax=axes[i]
        condition_df=df[df[condition_col]==j]
        #Creating Box Plot Per Field
        sns.boxplot(data=condition_df,x=field_col,y=feat_col,ax=ax,palette='Set2')
        sns.stripplot(data=condition_df,x=field_col,y=feat_col,ax=ax,color='black',alpha=0.3,size=2)
        feat=feat_col.replace('Value of Default', '').replace('_',' ')
        ax.set_title(f'{j}: {feat}',fontsize=12)
        ax.set_xlabel('Imaging Field',fontsize=11)
        ax.set_ylabel(feat,fontsize=11)
        ax.grid(True,alpha=0.3,axis='y')
    plt.tight_layout()
    plt.suptitle('Field-To-Field Consistency',fontsize=14,y=1.02)
    plt.show()

print("\nGenerating Field Consistency Plots For Each Feature...")
for i in feat_cols:
    plt_field_consistency(df_fig3,i)

Summarizing The Results

In [ ]:
def create_res_summary(test_res):
    print("Osmotic Stress Effects: Summary Of Findings")
    #Filtering Out Significant Results
    sig=test_res[test_res['Significant_Bonferroni']==True].copy()
    sig=sig.sort_values('Cohens_d',key=abs,ascending=False)
    print(f"\nSignificant Features (After Bonferroni Correction): {len(sig)}/{len(test_res)}")
    if len(sig)>0:
        print("\nTop Effects (Sorted By Effect Size):")
        for i,j in sig.iterrows():
            feat=j['Feature']
            diff=j['Mean_Difference']
            cohens_d=j['Cohens_d']
            p_val=j['P_value']
            eff=j['Effect_Size']
            direction="increased" if diff>0 else "decreased"

            print(f"\n{feat}:")
            print(f"Direction: {direction} In Hypertonic")
            print(f"Effect size: {cohens_d:.3f} ({eff})")
            print(f"P-value: {p_val:.4e}")
    else:
        print("\nNo Features Reached Bonferroni-Corrected Significance")
        print("Consider Using Less Conservative Correction (e.g., FDR) Or Examining")
        print("Features With Uncorrected p<0.05 And Medium-To-Large Effect Sizes.")

    #Features With Large Effect Sizes Even If Not Significant
    lrg_effs=test_res[abs(test_res['Cohens_d'])>0.5].copy()
    lrg_effs=lrg_effs.sort_values('Cohens_d',key=abs,ascending=False)
    print(f"\nFeatures With Medium-To-Large Effects (|d|>0.5): {len(lrg_effs)}")
    if len(lrg_effs)>0:
        print(lrg_effs[['Feature','Cohens_d','P_value','Effect_Size']].to_string(index=False))

create_res_summary(test_res)